# 02 — Training & Evaluation
Multimodal Hateful Meme Classifier (CLIP + Cross-Attention Fusion)

Run this notebook to train the model and evaluate on the dev set.

In [ ]:
import os, sys
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)   # needed so relative paths in config work

import yaml
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from transformers import CLIPProcessor

from src.datasets.meme_dataset import HatefulMemeDataset
from src.models.hateful_meme_model import HatefulMemeClassifier
from src.training.trainer import Trainer
from src.evaluation.metrics import compute_metrics, pretty_print_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Configuration

In [ ]:
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Optional notebook-level overrides ──────────────────────────────────
cfg['training']['num_epochs']  = 10    # change as needed
cfg['training']['batch_size']  = 32
cfg['training']['learning_rate'] = 1e-4

print(yaml.dump(cfg, default_flow_style=False))

## 2. Build Datasets & DataLoaders

In [ ]:
processor = CLIPProcessor.from_pretrained(cfg['model']['clip_model'])

train_ds = HatefulMemeDataset(
    jsonl_path=cfg['data']['train_path'],
    image_root=cfg['data']['image_root'],
    processor=processor,
    split='train',
)
dev_ds = HatefulMemeDataset(
    jsonl_path=cfg['data']['dev_path'],
    image_root=cfg['data']['image_root'],
    processor=processor,
    split='dev',
)

train_loader = DataLoader(
    train_ds, batch_size=cfg['training']['batch_size'],
    shuffle=True, num_workers=0, pin_memory=device.type=='cuda'
)
val_loader = DataLoader(
    dev_ds, batch_size=cfg['training']['batch_size'],
    shuffle=False, num_workers=0, pin_memory=device.type=='cuda'
)

print(f'Train: {len(train_ds):,} samples  ({len(train_loader)} batches)')
print(f'Dev  : {len(dev_ds):,} samples  ({len(val_loader)} batches)')

## 3. Initialise Model

In [ ]:
model = HatefulMemeClassifier(
    clip_model_name=cfg['model']['clip_model'],
    freeze_encoders=cfg['model']['freeze_encoders'],
    embed_dim=cfg['model']['embed_dim'],
    num_attention_heads=cfg['model']['num_attention_heads'],
    fusion_hidden_dim=cfg['model']['fusion_hidden_dim'],
    dropout=cfg['model']['dropout'],
)

p = model.count_parameters()
print(f"Parameters — total: {p['total']:,}  "
      f"trainable: {p['trainable']:,}  frozen: {p['frozen']:,}")

## 4. Train

In [ ]:
trainer_cfg = {
    **cfg['training'],
    'model_dir': cfg['paths']['model_dir'],
}

trainer = Trainer(model, train_loader, val_loader, trainer_cfg, device)
history = trainer.train()

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

# AUROC
axes[1].plot(epochs, history['val_auroc'], 'g-o')
axes[1].axhline(0.5, linestyle='--', color='gray', label='Random')
axes[1].set_title('Val AUROC'); axes[1].set_xlabel('Epoch'); axes[1].legend()

# F1 + Accuracy
axes[2].plot(epochs, history['val_f1'],       'm-o', label='F1')
axes[2].plot(epochs, history['val_accuracy'], 'c-o', label='Accuracy')
axes[2].set_title('Val F1 & Accuracy'); axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()

## 6. Evaluate Best Model on Dev Set

In [ ]:
# Load best checkpoint
ckpt = torch.load('outputs/models/best_model.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
print(f"Best checkpoint from epoch {ckpt['epoch']}")

metrics, logits, labels = trainer.evaluate_test(val_loader)
pretty_print_metrics(metrics, prefix='Best-model Dev')
print(metrics.get('report', ''))

## 7. Confusion Matrix

In [ ]:
import matplotlib.patches as mpatches

cm = np.array(metrics['confusion'])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred: Non-hateful', 'Pred: Hateful'])
ax.set_yticklabels(['True: Non-hateful', 'True: Hateful'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black',
                fontsize=16, fontweight='bold')
ax.set_title('Confusion Matrix — Dev Set')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150)
plt.show()

## 8. Error Analysis — Hard Examples

In [ ]:
import json
from PIL import Image
import torch.nn.functional as F

dev_data = [json.loads(l) for l in open('data/dev.jsonl')]
probs     = F.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
preds     = probs.argmax(axis=1)

# Find false positives (predicted hateful, actually not)
fp_idx = [i for i, (p, l) in enumerate(zip(preds, labels)) if p == 1 and l == 0]
# Find false negatives (predicted not hateful, actually hateful)
fn_idx = [i for i, (p, l) in enumerate(zip(preds, labels)) if p == 0 and l == 1]

print(f'False Positives: {len(fp_idx)}  |  False Negatives: {len(fn_idx)}')

def show_errors(indices, title, n=4):
    fig, axes = plt.subplots(1, min(n, len(indices)), figsize=(4*n, 4))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for ax, idx in zip(axes, indices[:n]):
        s = dev_data[idx]
        img = Image.open(os.path.join('data/img', os.path.basename(s['img']))).convert('RGB')
        ax.imshow(img)
        conf_hateful = probs[idx, 1]
        ax.set_title(
            f'True:{s["label"]} | P(hate):{conf_hateful:.2f}\n"{s["text"][:40]}"',
            fontsize=8
        )
        ax.axis('off')
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

show_errors(fp_idx, 'False Positives (predicted Hateful, actually NOT)')
show_errors(fn_idx, 'False Negatives (missed Hateful memes)')

---
## 9. Enhanced Run — Balanced Sampler + 20 Epochs (Phase 2)

This section retrains the **same cross-attention model** but with:
- `WeightedRandomSampler` (SMOTE-style class balancing — minority hateful class oversampled to match majority)
- 20 epochs (double Phase 1) with longer cosine warmup (200 steps)

Results saved separately to `outputs/models_smote/` so Phase 1 weights are untouched.

In [ ]:
import yaml
from src.training.sampler import make_balanced_sampler

with open('configs/config_smote.yaml') as f:
    cfg_smote = yaml.safe_load(f)

cfg_smote['training']['num_epochs'] = 20   # adjust on Kaggle if needed

processor_smote = CLIPProcessor.from_pretrained(cfg_smote['model']['clip_model'])

train_ds_smote = HatefulMemeDataset(
    jsonl_path=cfg_smote['data']['train_path'],
    image_root=cfg_smote['data']['image_root'],
    processor=processor_smote,
    split='train',
)
dev_ds_smote = HatefulMemeDataset(
    jsonl_path=cfg_smote['data']['dev_path'],
    image_root=cfg_smote['data']['image_root'],
    processor=processor_smote,
    split='dev',
)

# SMOTE-style balanced oversampling
sampler = make_balanced_sampler(train_ds_smote)

train_loader_smote = DataLoader(
    train_ds_smote, batch_size=cfg_smote['training']['batch_size'],
    sampler=sampler, num_workers=0, pin_memory=(device.type == 'cuda')
)
val_loader_smote = DataLoader(
    dev_ds_smote, batch_size=cfg_smote['training']['batch_size'],
    shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda')
)

model_smote = HatefulMemeClassifier(
    clip_model_name=cfg_smote['model']['clip_model'],
    freeze_encoders=cfg_smote['model']['freeze_encoders'],
    embed_dim=cfg_smote['model']['embed_dim'],
    num_attention_heads=cfg_smote['model']['num_attention_heads'],
    fusion_hidden_dim=cfg_smote['model']['fusion_hidden_dim'],
    dropout=cfg_smote['model']['dropout'],
)

trainer_cfg_smote = {
    **cfg_smote['training'],
    'model_dir': cfg_smote['paths']['model_dir'],
}

trainer_smote = Trainer(model_smote, train_loader_smote, val_loader_smote, trainer_cfg_smote, device)
history_smote = trainer_smote.train()

In [ ]:
# Phase 1 vs Phase 2 comparison
phase1_best_auroc = max(history['val_auroc'])
phase2_best_auroc = max(history_smote['val_auroc'])
phase1_best_acc   = max(history['val_accuracy'])
phase2_best_acc   = max(history_smote['val_accuracy'])

print("=" * 55)
print(f"{'Run':<30} {'AUROC':>8} {'Accuracy':>10}")
print("-" * 55)
print(f"{'Phase 1 (10ep, no sampler)':<30} {phase1_best_auroc:>8.4f} {phase1_best_acc:>9.1%}")
print(f"{'Phase 2 (20ep + SMOTE sampler)':<30} {phase2_best_auroc:>8.4f} {phase2_best_acc:>9.1%}")
print("=" * 55)

# Plot both training curves together
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
e1 = range(1, len(history['val_auroc']) + 1)
e2 = range(1, len(history_smote['val_auroc']) + 1)

axes[0].plot(e1, history['val_auroc'],       'b-o', label='Phase 1 (no sampler)')
axes[0].plot(e2, history_smote['val_auroc'], 'r-o', label='Phase 2 (SMOTE)')
axes[0].set_title('Val AUROC'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(e1, history['val_accuracy'],       'b-o', label='Phase 1 (no sampler)')
axes[1].plot(e2, history_smote['val_accuracy'], 'r-o', label='Phase 2 (SMOTE)')
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/phase_comparison.png', dpi=150)
plt.show()

---
## 10. Ablation Study — Fusion Strategy Comparison

Trains all four fusion variants on the **same config** so results are directly comparable.  
Mirrors Table 3 in the 2025 Elsevier paper format.

| Variant | Modalities | Fusion |
|---|---|---|
| `image_only` | Image | CLIP → MLP |
| `text_only` | Text | CLIP → MLP |
| `late_fusion` | Both | Concat → MLP (no attention) |
| `cross_attention` | Both | Bidirectional cross-attention → MLP (**ours**) |

> **Note**: This takes ~4× longer than a single run. On Kaggle T4 with 10 epochs each: ~2hrs.  
> To run only a subset, pass `variants = ['late_fusion', 'cross_attention']` below.

In [ ]:
from src.models.ablation_models import ImageOnlyClassifier, TextOnlyClassifier, LateFusionClassifier
import time, json, os, warnings
warnings.filterwarnings('ignore')

# ─── Config for ablation (10 epochs, no balanced sampler — isolates fusion effect) ───
with open('configs/config.yaml') as f:
    cfg_abl = yaml.safe_load(f)

cfg_abl['training']['num_epochs'] = 10   # change to 20 for fuller comparison

# Which variants to run — remove any you want to skip
variants = ['image_only', 'text_only', 'late_fusion', 'cross_attention']

def build_ablation_model(variant, cfg):
    mc = cfg['model']
    kw = dict(
        clip_model_name=mc['clip_model'],
        freeze_encoders=mc['freeze_encoders'],
        embed_dim=mc['embed_dim'],
        hidden_dim=mc['fusion_hidden_dim'],
        dropout=mc['dropout'],
    )
    if variant == 'image_only':     return ImageOnlyClassifier(**kw)
    if variant == 'text_only':      return TextOnlyClassifier(**kw)
    if variant == 'late_fusion':    return LateFusionClassifier(**kw)
    # cross_attention — full model
    return HatefulMemeClassifier(
        clip_model_name=mc['clip_model'],
        freeze_encoders=mc['freeze_encoders'],
        embed_dim=mc['embed_dim'],
        num_attention_heads=mc['num_attention_heads'],
        fusion_hidden_dim=mc['fusion_hidden_dim'],
        dropout=mc['dropout'],
    )

# Shared loaders (re-use existing ones)
ablation_results = {}

for variant in variants:
    print(f"\n### Variant: {variant.upper()} ###")
    t0 = time.time()

    m = build_ablation_model(variant, cfg_abl)
    abl_trainer_cfg = {
        **cfg_abl['training'],
        'model_dir': f'outputs/ablation/{variant}',
    }
    abl_trainer = Trainer(m, train_loader, val_loader, abl_trainer_cfg, device)
    hist = abl_trainer.train()

    best_auroc = max(hist['val_auroc'])
    best_epoch = hist['val_auroc'].index(best_auroc)
    best_acc   = hist['val_accuracy'][best_epoch]
    best_f1    = hist['val_f1'][best_epoch]
    best_f1m   = hist.get('val_f1_macro', [0.0]*len(hist['val_auroc']))[best_epoch]

    ablation_results[variant] = {
        'best_val_auroc': best_auroc,
        'best_val_accuracy': best_acc,
        'best_val_f1': best_f1,
        'best_val_f1_macro': best_f1m,
        'runtime_s': round(time.time() - t0, 1),
    }
    print(f"  AUROC={best_auroc:.4f}  Acc={best_acc:.1%}  Macro-F1={best_f1m:.4f}  ({ablation_results[variant]['runtime_s']}s)")

# Save
os.makedirs('outputs/ablation', exist_ok=True)
with open('outputs/ablation/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

print("\nAblation results saved.")

In [ ]:
# Print ablation comparison table
print("=" * 72)
print("  ABLATION STUDY — Fusion Strategy Comparison (Table)")
print("=" * 72)
print(f"  {'Variant':<22} {'Val AUROC':>10} {'Val Acc':>9} {'Macro-F1':>10} {'Time(s)':>8}")
print("-" * 72)
for variant, r in ablation_results.items():
    marker = " ◀ OURS" if variant == 'cross_attention' else ""
    print(f"  {variant:<22} {r['best_val_auroc']:>10.4f} {r['best_val_accuracy']:>8.1%} "
          f"{r['best_val_f1_macro']:>10.4f} {r['runtime_s']:>8.1f}{marker}")
print("=" * 72)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(ablation_results.keys())
aurocs = [ablation_results[v]['best_val_auroc'] for v in names]
accs   = [ablation_results[v]['best_val_accuracy'] for v in names]
colors = ['#4878d0', '#ee854a', '#6acc65', '#d65f5f']

bars1 = axes[0].bar(names, aurocs, color=colors, edgecolor='black', linewidth=0.7)
axes[0].set_title('Val AUROC by Fusion Variant'); axes[0].set_ylabel('AUROC')
axes[0].set_ylim(0.5, 0.85)
axes[0].axhline(0.5, linestyle='--', color='gray', linewidth=0.8, label='Random')
for bar, val in zip(bars1, aurocs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

bars2 = axes[1].bar(names, [a * 100 for a in accs], color=colors, edgecolor='black', linewidth=0.7)
axes[1].set_title('Val Accuracy (%) by Fusion Variant'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(50, 90)
for bar, val in zip(bars2, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/ablation_comparison.png', dpi=150)
plt.show()

---
## 11. Package & Download Results

Zips everything into a single `all_results.zip` for download.

In [ ]:
import shutil, os, json
from IPython.display import FileLink

# ── Collect everything worth keeping ─────────────────────────────────────────
os.makedirs("all_results", exist_ok=True)

# 1. Phase 1 best model + history
if os.path.exists("outputs/models"):
    shutil.copytree("outputs/models", "all_results/phase1_models", dirs_exist_ok=True)

# 2. Phase 2 (SMOTE) best model + history
if os.path.exists("outputs/models_smote"):
    shutil.copytree("outputs/models_smote", "all_results/phase2_smote_models", dirs_exist_ok=True)

# 3. Ablation results (JSON + model checkpoints)
if os.path.exists("outputs/ablation"):
    shutil.copytree("outputs/ablation", "all_results/ablation", dirs_exist_ok=True)

# 4. All saved plots
for fname in ["training_curves.png", "confusion_matrix.png",
              "phase_comparison.png", "ablation_comparison.png"]:
    src = os.path.join("outputs", fname)
    if os.path.exists(src):
        shutil.copy(src, "all_results/")

# 5. Write a summary JSON
summary = {}
for tag, hist_obj in [("phase1", history), ("phase2_smote", history_smote)]:
    try:
        summary[tag] = {
            "best_val_auroc":    max(hist_obj["val_auroc"]),
            "best_val_accuracy": max(hist_obj["val_accuracy"]),
            "best_epoch":        hist_obj["val_auroc"].index(max(hist_obj["val_auroc"])) + 1,
            "num_epochs":        len(hist_obj["val_auroc"]),
        }
    except Exception:
        pass

if ablation_results:
    summary["ablation"] = ablation_results

with open("all_results/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary:")
print(json.dumps(summary, indent=2))

# ── Zip and offer download link ───────────────────────────────────────────────
shutil.make_archive("all_results", "zip", ".", "all_results")
print("\nZip created: all_results.zip")
FileLink("all_results.zip")